# บทที่ 13 - หน่วยความจำตัวแทนกับกราฟความรู้ Cognee


## การตั้งค่า

สมุดบันทึกนี้สาธิตวิธีสร้าง **ผู้ช่วยเขียนโค้ดอัจฉริยะ** ที่มีหน่วยความจำถาวรโดยใช้แผนภูมิความรู้ [**Cognee**](https://www.cognee.ai/) และ **Microsoft Agent Framework** (MAF)

Cognee แปลงข้อความที่ไม่มีโครงสร้างให้เป็นแผนภูมิความรู้ที่มีโครงสร้าง สามารถสืบค้นได้ โดยอาศัยเวกเตอร์ฝัง — มอบหน่วยความจำระยะยาวที่มีความสัมพันธ์ลึกซึ้งสำหรับเอเย่นต์ของคุณ

### สิ่งที่คุณจะได้เรียนรู้
1. **สร้างแผนภูมิความรู้**: แปลงโปรไฟล์นักพัฒนาและแนวปฏิบัติที่ดีที่สุดให้เป็นความรู้ที่มีโครงสร้างและสืบค้นได้
2. **ผสาน Cognee กับ MAF**: ใช้ฟังก์ชัน `@tool` เพื่อให้เอเย่นต์ MAF สืบค้นแผนภูมิความรู้ของ Cognee
3. **บทสนทนาที่รับรู้บริบทในเซสชัน**: รักษาบริบทระหว่างคำถามหลายคำถามในเซสชันเดียวกัน
4. **หน่วยความจำระยะยาว**: เก็บรักษาความรู้สำคัญในระหว่างเซสชันและดึงมาใช้ในการสนทนาใหม่

### ข้อกำหนดเบื้องต้น
- Python 3.9+
- Redis รันอยู่ในเครื่อง (`docker run -d -p 6379:6379 redis`) สำหรับการจัดการเซสชัน
- คีย์ API สำหรับ LLM (เช่น OpenAI) — ตั้งค่า `LLM_API_KEY` ในไฟล์ `.env`
- ตั้งค่า `CACHING=true` ในไฟล์ `.env` (จำเป็นสำหรับเซสชัน Cognee)
- โปรเจกต์ Microsoft Foundry ที่มีโมเดลแชทถูก deploy แล้ว
- ตั้งค่า `AZURE_AI_PROJECT_ENDPOINT` และ `AZURE_AI_MODEL_DEPLOYMENT_NAME` ในไฟล์ `.env`
- ลงชื่อเข้าใช้ Azure CLI แล้ว (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ประเภทของหน่วยความจำของเอเย่นต์

โน้ตบุ๊กนี้สำรวจประเภทหน่วยความจำสามประเภทเดียวกันกับในโน้ตบุ๊กบทเรียนหลักบทที่ 13 แต่ใช้ Cognee เป็นแบ็คเอนด์หน่วยความจำระยะยาว:

| ประเภทหน่วยความจำ | กลไก | ระยะเวลา |
|---|---|---|
| **หน่วยความจำทำงาน** | `agent.create_session()` (MAF) | การสนทนาเดียว |
| **หน่วยความจำระยะสั้น** | แคชเซสชัน Cognee (Redis) | เซสชันเดียว |
| **หน่วยความจำระยะยาว** | กราฟความรู้ Cognee + เวกเตอร์ | ถาวร |

### สถาปัตยกรรมหน่วยความจำของ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## เตรียมที่เก็บ Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ส่วนที่ 1 — การสร้างฐานความรู้

เรารวบรวมข้อมูลสามประเภทเพื่อสร้างฐานความรู้ที่ครอบคลุมสำหรับผู้ช่วยโค้ดดิ้งของเรา:

1. **โปรไฟล์นักพัฒนา** — ความชำนาญส่วนบุคคลและพื้นหลังทางเทคนิค
2. **แนวปฏิบัติที่ดีที่สุดของ Python** — Zen ของ Python พร้อมแนวทางปฏิบัติที่เป็นประโยชน์
3. **บทสนทนาในอดีต** — การถามตอบระหว่างนักพัฒนาและผู้ช่วย AI ในอดีต


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## แสดงภาพกราฟความรู้

Cognee สามารถแสดงภาพอินเทอร์แอคทีฟในรูปแบบ HTML ของเอนทิตีและความสัมพันธ์ที่มันสกัดออกมาได้


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## เติมเต็มความจำด้วย Memify

`memify()` วิเคราะห์กราฟความรู้และสร้างกฎอัจฉริยะ — ระบุรูปแบบ แนวทางปฏิบัติที่ดีที่สุด และความสัมพันธ์ระหว่างแนวคิดต่าง ๆ


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ส่วนที่ 2 — ตัวแทน MAF กับ Cognee Tools

ตอนนี้เราจะสร้างตัวแทน MAF ที่สามารถสืบค้นกราฟความรู้ของ Cognee ผ่านฟังก์ชัน `@tool` ซึ่งช่วยให้ตัวแทนใช้ประโยชน์จากพลังเต็มที่ของการค้นหาเชิงความหมายที่รับรู้กราฟ ในขณะเดียวกันก็รักษาบริบทการสนทนาผ่านเซสชันได้


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## หน่วยความจำชั่วคราวกับเซสชัน

`AgentSession` (สร้างผ่าน `agent.create_session()`) ให้หน่วยความจำชั่วคราวภายในเซสชัน ตัวแทนสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าได้ในขณะที่ยังสามารถค้นหาฐานความรู้ระยะยาวของ Cognee ได้


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## เซสชันใหม่ — ความจำระยะยาวยังคงอยู่

การเริ่มเซสชันใหม่จะล้างหน่วยความจำที่ใช้ในงาน แต่กราฟความรู้ Cognee ยังพร้อมใช้งานอยู่ ตัวแทนสามารถดึงความรู้ระยะยาวเดิมออกมาได้ในบทสนทนาใหม่ทั้งหมด


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในโน้ตบุ๊กนี้คุณได้สร้างผู้ช่วยเขียนโค้ดที่รวม **หน่วยความจำการทำงานของ MAF** (`agent.create_session()`) กับ **กราฟความรู้ระยะยาวของ Cognee** เข้าด้วยกัน

### สิ่งที่คุณได้เรียนรู้
1. **การสร้างกราฟความรู้**: Cognee รับข้อมูลข้อความที่ไม่มีโครงสร้างและสร้างกราฟพร้อมหน่วยความจำรูปเวกเตอร์
2. **การเติมเต็มกราฟด้วย memify**: ข้อเท็จจริงที่ได้มาและความสัมพันธ์ที่ลึกซึ้งขึ้นบนกราฟที่มีอยู่ของคุณ
3. **การผสานรวม MAF + Cognee**: ฟังก์ชัน `@tool` ให้เอเย่นต์ MAF สอบถามกราฟของ Cognee อย่างเป็นธรรมชาติ
4. **หน่วยความจำการทำงาน + หน่วยความจำระยะยาว**: `AgentSession` (ผ่าน `agent.create_session()`) ให้บริบทของเซสชันในขณะที่ Cognee ให้ความรู้ถาวร
5. **การค้นหากรองด้วย NodeSets**: เจาะจงกลุ่มย่อยเฉพาะของกราฟความรู้ (เช่น เฉพาะหลักการ)

### ข้อสรุปสำคัญ
- **Cognee** เปลี่ยนข้อความดิบเป็นหน่วยความจำที่มีโครงสร้างและรับรู้ความสัมพันธ์ — มีพลังมากกว่าร้านเก็บเวกเตอร์แบบแบน
- ฟังก์ชัน **`@tool`** เป็นสะพานเชื่อมเอเย่นต์ MAF กับระบบความรู้ภายนอกอย่างราบรื่น
- **`AgentSession`** (ผ่าน `agent.create_session()`) แยกบริบทต่อบทสนทนาออกจากความรู้ที่มีอายุยาวนาน
- กราฟความรู้เดียวกันให้บริการหลายเซสชันและเอเย่นต์

### การใช้งานในโลกจริง
- **ผู้ช่วยนักพัฒนา**: ตรวจสอบโค้ด, วิเคราะห์เหตุการณ์, ผู้ช่วยสถาปัตยกรรม
- **ผู้ช่วยฝ่ายลูกค้า**: เจ้าหน้าที่สนับสนุนเหนือเอกสารผลิตภัณฑ์, คำถามที่พบบ่อย, และบันทึก CRM
- **ผู้ช่วยผู้เชี่ยวชาญภายใน**: ผู้ช่วยนโยบาย, กฎหมาย หรือความปลอดภัยที่ตรรกะบนแนวทาง
- **ชั้นข้อมูลรวมกัน**: รวมข้อมูลมีโครงสร้างและไม่มีโครงสร้างเป็นกราฟที่สามารถสืบค้นได้

### ขั้นตอนถัดไป
- ทดลองใช้งานความตระหนักตามเวลาใน Cognee
- กำหนดออนโทโลยี OWL สำหรับคุณภาพกราฟเฉพาะโดเมน
- เพิ่มวงจรการตอบกลับของผู้ใช้เพื่อปรับปรุงการดึงข้อมูลเมื่อเวลาผ่านไป
- ขยายไปสู่ระบบหลายเอเย่นต์ที่แชร์ชั้นความจำ Cognee เดียวกัน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
